In [1]:
# ============================================================================
# GPT-2 Fine-tuning Diagnostic Script
# ============================================================================
# Run this BEFORE training to check for potential issues

import os
import json
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import sentencepiece as spm

# ============================================================================
# CONFIGURATION (Copy from your main script)
# ============================================================================

class Config:
    CHECKPOINT_PATH = r"C:\Users\prash\Documents\AI\Major Project\log\bpe16-models\model_03299.pt"
    TOKENIZER_PATH = r"C:\Users\prash\Documents\AI\Major Project\bpe-token-models\bpe-16.model"
    TRAIN_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\train_plus_val.jsonl"
    TEST_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\nepali_test.jsonl"
    
    MAX_SOURCE_LENGTH = 384
    MAX_TARGET_LENGTH = 128
    BATCH_SIZE = 4
    FREEZE_EMBEDDINGS = False
    FREEZE_BLOCKS = list(range(8))

config = Config()
device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================================
# MODEL ARCHITECTURE (Minimal - just for loading)
# ============================================================================

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 16384
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, input_ids, labels=None):
        B, T = input_ids.size()
        assert T <= self.config.block_size
        pos = torch.arange(0, T, dtype=torch.long, device=input_ids.device)
        pos_emb = self.transformer.wpe(pos)
        tok_emb = self.transformer.wte(input_ids)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        return loss, logits

# ============================================================================
# DATASET (Minimal)
# ============================================================================

PROMPT_TEMPLATE = "Summarize the following article:\n{text}\nSummary:"

class SummarizationDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_src, max_tgt):
        self.tokenizer = tokenizer
        self.max_src = max_src
        self.max_tgt = max_tgt
        self.data = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    item = json.loads(line)
                    if 'text' in item and 'summary' in item:
                        self.data.append(item)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        prompt = PROMPT_TEMPLATE.format(text=item['text'])
        full_text = prompt + item['summary']
        tokens = self.tokenizer.encode(full_text)
        if len(tokens) > self.max_src + self.max_tgt:
            tokens = tokens[:self.max_src + self.max_tgt]
        input_ids = torch.tensor(tokens, dtype=torch.long)
        labels = input_ids.clone()
        prompt_len = len(self.tokenizer.encode(prompt))
        labels[:prompt_len] = -100
        return {'input_ids': input_ids, 'labels': labels, 'text': item['text'], 'summary': item['summary']}

# ============================================================================
# DIAGNOSTIC CHECKS
# ============================================================================

print("="*80)
print("GPT-2 FINE-TUNING DIAGNOSTIC SCRIPT")
print("="*80)

# ============================================================================
# 1. LOAD MODEL & TOKENIZER
# ============================================================================

print("\n" + "="*80)
print("1. LOADING MODEL & TOKENIZER")
print("="*80)

sp = spm.SentencePieceProcessor()
sp.load(config.TOKENIZER_PATH)
print(f"✓ Tokenizer loaded (vocab: {sp.vocab_size()})")

checkpoint = torch.load(config.CHECKPOINT_PATH, map_location=device, weights_only=False)
model_config = checkpoint['config']
model = GPT(model_config)
model.load_state_dict(checkpoint['model'])
model.to(device)
print(f"✓ Model loaded (step: {checkpoint['step']}, val_loss: {checkpoint['val_loss']:.4f})")

# ============================================================================
# 2. CHECK WEIGHT TYING
# ============================================================================

print("\n" + "="*80)
print("2. WEIGHT TYING CHECK")
print("="*80)

wte_id = id(model.transformer.wte.weight)
lm_head_id = id(model.lm_head.weight)

if wte_id == lm_head_id:
    print("⚠️  WARNING: Weight tying is ACTIVE")
    print("   wte.weight and lm_head.weight share the same memory")
    print("   This can cause NaN if one is frozen and the other isn't")
    print("\n   RECOMMENDATION: Break weight tying with:")
    print("   model.lm_head.weight = nn.Parameter(model.transformer.wte.weight.clone().detach())")
else:
    print("✓ Weight tying is NOT active (weights are independent)")

# ============================================================================
# 3. CHECK MODEL WEIGHTS FOR NaN/Inf
# ============================================================================

print("\n" + "="*80)
print("3. MODEL WEIGHT SANITY CHECK")
print("="*80)

issues_found = False
for name, param in model.named_parameters():
    has_nan = torch.isnan(param).any().item()
    has_inf = torch.isinf(param).any().item()
    
    if has_nan or has_inf:
        issues_found = True
        if has_nan:
            print(f"⚠️  NaN detected in {name}")
        if has_inf:
            print(f"⚠️  Inf detected in {name}")

if not issues_found:
    print("✓ No NaN or Inf values found in model weights")

# Show statistics for trainable parameters
print("\nTrainable parameter statistics:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name:50s} | min: {param.min().item():8.4f} | max: {param.max().item():8.4f} | mean: {param.mean().item():8.4f}")

# ============================================================================
# 4. APPLY FREEZING & CHECK
# ============================================================================

print("\n" + "="*80)
print("4. FREEZING STRATEGY CHECK")
print("="*80)

if config.FREEZE_EMBEDDINGS:
    for param in model.transformer.wte.parameters():
        param.requires_grad = False
    for param in model.transformer.wpe.parameters():
        param.requires_grad = False
    print("✓ Embeddings frozen")
else:
    for param in model.transformer.wpe.parameters():
        param.requires_grad = False
    print("✓ Position embeddings frozen, token embeddings trainable")

for block_idx in config.FREEZE_BLOCKS:
    for param in model.transformer.h[block_idx].parameters():
        param.requires_grad = False
print(f"✓ Blocks {config.FREEZE_BLOCKS[0]}-{config.FREEZE_BLOCKS[-1]} frozen")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

# Check for frozen/trainable conflicts with weight tying
if wte_id == lm_head_id:
    wte_trainable = model.transformer.wte.weight.requires_grad
    lm_head_trainable = model.lm_head.weight.requires_grad
    if wte_trainable != lm_head_trainable:
        print("\n⚠️  CRITICAL: Weight tying conflict detected!")
        print(f"   wte.weight.requires_grad = {wte_trainable}")
        print(f"   lm_head.weight.requires_grad = {lm_head_trainable}")
        print("   This WILL cause NaN losses!")

# ============================================================================
# 5. CHECK DATASET
# ============================================================================

print("\n" + "="*80)
print("5. DATASET SANITY CHECK")
print("="*80)

train_dataset = SummarizationDataset(config.TRAIN_JSONL, sp, config.MAX_SOURCE_LENGTH, config.MAX_TARGET_LENGTH)
test_dataset = SummarizationDataset(config.TEST_JSONL, sp, config.MAX_SOURCE_LENGTH, config.MAX_TARGET_LENGTH)

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Test samples:  {len(test_dataset)}")

print("\nChecking first 10 training samples:")
data_issues = []
for i in range(min(10, len(train_dataset))):
    item = train_dataset[i]
    input_ids = item['input_ids']
    labels = item['labels']
    
    # Checks
    seq_len = len(input_ids)
    non_masked = (labels != -100).sum().item()
    min_token = input_ids.min().item()
    max_token = input_ids.max().item()
    
    issues = []
    if seq_len > model.config.block_size:
        issues.append(f"seq_len={seq_len} > block_size={model.config.block_size}")
    if non_masked == 0:
        issues.append("all labels masked")
    if max_token >= model.config.vocab_size:
        issues.append(f"token_id={max_token} >= vocab_size={model.config.vocab_size}")
    if min_token < 0:
        issues.append(f"negative token_id={min_token}")
    
    if issues:
        print(f"  Sample {i}: ⚠️  {', '.join(issues)}")
        data_issues.append(i)
    else:
        print(f"  Sample {i}: ✓ len={seq_len}, non_masked={non_masked}, tokens=[{min_token}, {max_token}]")

if data_issues:
    print(f"\n⚠️  Found issues in {len(data_issues)} samples")
else:
    print(f"\n✓ All checked samples look good")

# ============================================================================
# 6. TEST FORWARD PASS
# ============================================================================

print("\n" + "="*80)
print("6. TEST FORWARD PASS")
print("="*80)

def collate_fn(batch):
    max_len = max(len(item['input_ids']) for item in batch)
    input_ids = []
    labels = []
    for item in batch:
        seq_len = len(item['input_ids'])
        pad_len = max_len - seq_len
        input_ids.append(torch.cat([item['input_ids'], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([item['labels'], torch.full((pad_len,), -100, dtype=torch.long)]))
    return {'input_ids': torch.stack(input_ids), 'labels': torch.stack(labels)}

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print("Testing forward pass on first 5 batches...")
model.train()

for batch_idx, batch in enumerate(train_loader):
    if batch_idx >= 5:
        break
    
    input_ids = batch['input_ids'].to(device)
    labels = batch['labels'].to(device)
    
    try:
        with torch.no_grad():
            loss, logits = model(input_ids, labels)
        
        has_nan = torch.isnan(loss).item()
        has_inf = torch.isinf(loss).item()
        
        if has_nan or has_inf:
            print(f"  Batch {batch_idx}: ⚠️  loss={'NaN' if has_nan else 'Inf'}")
        else:
            print(f"  Batch {batch_idx}: ✓ loss={loss.item():.4f}")
    except Exception as e:
        print(f"  Batch {batch_idx}: ⚠️  ERROR: {str(e)}")

# ============================================================================
# 7. TEST BACKWARD PASS
# ============================================================================

print("\n" + "="*80)
print("7. TEST BACKWARD PASS")
print("="*80)

print("Testing backward pass on first batch...")

batch = next(iter(train_loader))
input_ids = batch['input_ids'].to(device)
labels = batch['labels'].to(device)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-5)

try:
    loss, _ = model(input_ids, labels)
    print(f"  Forward pass: loss={loss.item():.4f}")
    
    loss.backward()
    print(f"  Backward pass: ✓")
    
    # Check gradients
    total_norm = 0.0
    max_grad = 0.0
    nan_grads = []
    inf_grads = []
    
    for name, param in model.named_parameters():
        if param.grad is not None:
            param_norm = param.grad.data.norm(2).item()
            total_norm += param_norm ** 2
            max_grad = max(max_grad, param.grad.abs().max().item())
            
            if torch.isnan(param.grad).any():
                nan_grads.append(name)
            if torch.isinf(param.grad).any():
                inf_grads.append(name)
    
    total_norm = total_norm ** 0.5
    
    print(f"  Gradient norm: {total_norm:.4f}")
    print(f"  Max gradient value: {max_grad:.4f}")
    
    if nan_grads:
        print(f"  ⚠️  NaN gradients in: {nan_grads}")
    if inf_grads:
        print(f"  ⚠️  Inf gradients in: {inf_grads}")
    
    if not nan_grads and not inf_grads:
        print(f"  ✓ All gradients are valid")
    
    # Gradient norm interpretation
    if total_norm > 100:
        print(f"  ⚠️  WARNING: Very large gradient norm ({total_norm:.1f})")
        print(f"     Consider lower learning rate or stricter clipping")
    elif total_norm > 10:
        print(f"  ⚠️  Moderate gradient norm ({total_norm:.1f})")
        print(f"     Monitor for potential instability")
    else:
        print(f"  ✓ Gradient norm looks reasonable ({total_norm:.1f})")
        
except Exception as e:
    print(f"  ⚠️  ERROR during backward: {str(e)}")

# ============================================================================
# SUMMARY & RECOMMENDATIONS
# ============================================================================

print("\n" + "="*80)
print("DIAGNOSTIC SUMMARY & RECOMMENDATIONS")
print("="*80)

recommendations = []

# Check weight tying
if wte_id == lm_head_id:
    wte_trainable = model.transformer.wte.weight.requires_grad
    lm_head_trainable = model.lm_head.weight.requires_grad
    if wte_trainable != lm_head_trainable:
        recommendations.append({
            'severity': 'CRITICAL',
            'issue': 'Weight tying conflict',
            'fix': 'model.lm_head.weight = nn.Parameter(model.transformer.wte.weight.clone().detach())'
        })
    elif wte_trainable and lm_head_trainable:
        recommendations.append({
            'severity': 'WARNING',
            'issue': 'Weight tying with both trainable (double gradients)',
            'fix': 'Consider breaking weight tying OR lowering learning rate to 1e-5'
        })

# Check gradient norm
if 'total_norm' in locals() and total_norm > 10:
    recommendations.append({
        'severity': 'WARNING',
        'issue': f'Large gradient norm ({total_norm:.1f})',
        'fix': 'LEARNING_RATE = 1e-5 or MAX_GRAD_NORM = 0.5'
    })

# Check data issues
if data_issues:
    recommendations.append({
        'severity': 'WARNING',
        'issue': f'Data issues in {len(data_issues)} samples',
        'fix': 'Review dataset processing and tokenization'
    })

if recommendations:
    print("\n⚠️  ISSUES FOUND:\n")
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. [{rec['severity']}] {rec['issue']}")
        print(f"   Fix: {rec['fix']}\n")
else:
    print("\n✓ No major issues detected!")
    print("  Your model should train without NaN losses.")

print("="*80)
print("DIAGNOSTIC COMPLETE")
print("="*80)

GPT-2 FINE-TUNING DIAGNOSTIC SCRIPT

1. LOADING MODEL & TOKENIZER
✓ Tokenizer loaded (vocab: 16384)
✓ Model loaded (step: 3299, val_loss: 3.0820)

2. WEIGHT TYING CHECK
⚠️  WARNING: Weight tying is ACTIVE
   wte.weight and lm_head.weight share the same memory
   This can cause NaN if one is frozen and the other isn't

   RECOMMENDATION: Break weight tying with:
   model.lm_head.weight = nn.Parameter(model.transformer.wte.weight.clone().detach())

3. MODEL WEIGHT SANITY CHECK
✓ No NaN or Inf values found in model weights

Trainable parameter statistics:
  transformer.wte.weight                             | min:  -0.2263 | max:   0.2287 | mean:  -0.0001
  transformer.wpe.weight                             | min:  -0.2178 | max:   0.2005 | mean:   0.0000
  transformer.h.0.ln_1.weight                        | min:   0.8314 | max:   0.9726 | mean:   0.8935
  transformer.h.0.ln_1.bias                          | min:  -0.0230 | max:   0.0190 | mean:   0.0001
  transformer.h.0.attn.c_attn.wei